![Redis](https://redis.io/wp-content/uploads/2024/04/Logotype.svg?auto=webp&quality=85,75&width=120)

# Redis Online Feature Store with Featureform

In this recipe, we will learn how to define, materialize, and serve machine learning features with [**Featureform**](https://docs.featureform.com/) using **Redis** as the low-latency [online (inference) store](https://redis.io/docs/latest/develop/ai/featureform/).

## The problem: features are where ML projects break
Most of the effort in a production ML system is not the model — it's the **features**. A feature like "average transaction amount over the last 30 days" has to be computed one way in a batch job to train the model, and a *different* way in application code to score a live request. When those two implementations drift apart you get **training-serving skew**: the model was trained on numbers it never actually sees in production, and accuracy quietly degrades. On top of that, the same feature gets re-implemented by every team that needs it, nobody can find what already exists, and there is no record of how any given value was produced.

## What a feature store gives you
A **feature store** is the interface between your raw data and your models. It lets you:
- **Define a feature once** and serve the *identical* computation to both training (offline, high-throughput) and inference (online, low-latency) — killing training-serving skew.
- **Reuse and discover** features across models and teams instead of rebuilding them.
- **Version and audit** feature definitions alongside model code, so every prediction is traceable to the exact logic that produced it.

## Why Featureform specifically
[Featureform](https://docs.featureform.com/) is a **virtual** feature store — it does not copy your data into a new monolithic system. Instead it:
- **Leaves your data where it is.** Register your existing warehouse, database, or stream as a *provider*; there is no migration.
- **Treats features as code.** Transformations are plain Python/SQL functions, versioned in git and reviewed like any other code — not click-ops in a UI.
- **Separates definition from infrastructure.** The same feature definition can be materialized to different online stores; you pick the right engine for each job.
- **Tracks lineage.** Every feature, label, and training set is a named, versioned resource with a recorded path back to its source.

## Why Redis as the online store
Training reads happen in bulk and can be slow; **inference reads happen one entity at a time, on the critical path of a live request**, and must be fast. Redis is a natural fit as Featureform's inference store: in-memory, sub-millisecond point lookups, and horizontally scalable — so a fraud model can fetch a user's features and score a transaction well within a request budget. Featureform computes features in the offline store (here, Postgres) and **materializes** the results into Redis for serving.

## What we'll build
The canonical Featureform fraud-detection quickstart, wired to serve from Redis:
- **Postgres** as the **offline store** — holds the raw transaction history and runs the feature transformations.
- **Redis** as the **inference store** — materialized feature values are pushed here and served at sub-millisecond latency.

We'll register both, define an `avg_transactions` feature and a `fraudulent` label keyed by user, build a training set, materialize to Redis, and serve a live feature lookup.

## Running this notebook

> ⚠️ **This notebook runs locally with Docker — it will not run on Google Colab or in notebook CI**, because Featureform needs a running coordinator server plus provider containers (there is no Docker daemon on Colab).

**Prerequisites**
1. [Install Docker](https://docs.docker.com/get-docker/) and make sure the daemon is running (`docker ps`).
2. Install the Featureform CLI and start the quickstart stack from a terminal:

   ```bash
   pip install featureform
   featureform deploy docker --quickstart
   ```

   This pulls and starts three containers: the **Featureform** coordinator (gRPC on `localhost:7878`, dashboard on `http://localhost`), a **Redis** container (online store, published on `localhost:6379`), and a **Postgres** container pre-loaded with an example `Transactions` table (offline store, `localhost:5432`).
3. Run the cells below in order. Tear everything down at the end with `featureform stop docker`.

The deploy step is also runnable from the notebook (next cell) if you prefer.

## Environment Setup

### Install Python Dependencies

In [1]:
%pip install -q featureform redis pandas

Note: you may need to restart the kernel to use updated packages.


### Start Featureform, Redis, and Postgres

Skip this cell if you already ran `featureform deploy docker --quickstart` in a terminal (see above).

In [2]:
# NBVAL_SKIP
!featureform deploy docker --quickstart

Deploying Featureform on Docker
Starting Docker deployment on Darwin 24.6.0
Checking if featureform container exists...
	Container featureform has status "running"
	Container featureform is already running. Skipping...
Checking if quickstart-postgres container exists...
	Container quickstart-postgres has status "running"
	Container quickstart-postgres is already running. Skipping...
Checking if quickstart-redis container exists...
	Container quickstart-redis has status "running"
	Container quickstart-redis is already running. Skipping...

Pulling Quickstart files
	Pulling definitions.py
		definitions.py already exists. Skipping...
	Pulling serving.py
		serving.py already exists. Skipping...
	Pulling training.py
		training.py already exists. Skipping...

Featureform is now running!
To access the dashboard, visit http://localhost:80
Run jupyter notebook in the quickstart directory to get started.


### Point the client at the Featureform server

The Python client talks to the coordinator over gRPC on `localhost:7878`. `insecure=True` is required because the quickstart container serves an unencrypted endpoint.

In [3]:
import os

FEATUREFORM_HOST = os.getenv("FEATUREFORM_HOST", "localhost:7878")

## Register providers

We register Redis as the inference store and Postgres as the offline store.

The `host` is the address at which the **Featureform coordinator container** reaches each provider. The quickstart publishes Redis and Postgres on the host machine, so the coordinator reaches them via `host.docker.internal`.

> **On Linux**, `host.docker.internal` may not resolve — use the Docker bridge IP `172.17.0.1` instead (see [featureform#1156](https://github.com/featureform/featureform/issues/1156)).
>
> **Using your own Redis?** Swap `REDIS_HOST` / `REDIS_PORT` / `REDIS_PASSWORD` below for your [Redis Cloud](https://redis.io/cloud/) or Redis Enterprise endpoint. Featureform will materialize and serve features from that instance instead.

In [4]:
import featureform as ff

# Address the Featureform *coordinator container* uses to reach the providers.
# Mac/Windows: "host.docker.internal".  Linux: try "172.17.0.1".
PROVIDER_HOST = os.getenv("PROVIDER_HOST", "host.docker.internal")

REDIS_HOST = os.getenv("REDIS_HOST", PROVIDER_HOST)
REDIS_PORT = int(os.getenv("REDIS_PORT", "6379"))
REDIS_PASSWORD = os.getenv("REDIS_PASSWORD", "")

redis = ff.register_redis(
    name="redis-quickstart",
    description="Redis online (inference) store",
    host=REDIS_HOST,
    port=REDIS_PORT,
    password=REDIS_PASSWORD,
    db=0,
)

postgres = ff.register_postgres(
    name="postgres-quickstart",
    description="Postgres offline store with example transaction data",
    host=PROVIDER_HOST,
    port="5432",
    user="postgres",
    password="password",
    database="postgres",
)

## Register the source data

The Postgres quickstart image ships a `Transactions` table. We register it as a source so features can be derived from it.

In [5]:
transactions = postgres.register_table(
    name="transactions",
    variant="quickstart",
    table="transactions",
)

## Define a feature transformation

Features in Featureform are just transformations over registered sources. Here we compute each user's **average transaction amount** with a SQL transformation that runs in the offline store (Postgres). The `{{transactions.quickstart}}` placeholder references the source we just registered.

*Why this matters:* this decorated function **is** the single definition of the feature. It runs in Postgres to build training data and its output is materialized to Redis for serving. It's one piece of code, so the two can never drift. It's versioned in git and reviewable like any other function, and the `variant="quickstart"` tag lets you evolve the logic later without breaking models pinned to the old version.

In [6]:
@postgres.sql_transformation(variant="quickstart")
def average_user_transaction():
    """Average transaction amount per user, computed in the offline store."""
    return (
        "SELECT CustomerID as user_id, avg(TransactionAmount) as avg_transaction_amt "
        "FROM {{transactions.quickstart}} GROUP BY user_id"
    )

## Define the entity, feature, and label

These three resource types are the core of how Featureform models data, and they play distinct roles:

- **Entity** — *what a row is about.* Here the entity is a **user**, declared with `@ff.entity`. It's the join key: every feature and label below is keyed by user, so Featureform knows how to line them up. At serving time you look features up by an entity key (`{"user": "C1214240"}`).
- **Feature** — *a model input.* `avg_transactions` is a per-user value derived from the `average_user_transaction` transformation. Because it has `inference_store=redis`, its values are **materialized into Redis** and served online, one entity at a time, on the request path.
- **Label** — *the prediction target.* `fraudulent` (from the `isfraud` column) is the ground truth the model learns to predict. Labels are used **only offline** to build training sets; they are never materialized to the online store, because at inference time the answer is exactly what you're trying to produce.

**How they interact:** the entity is the glue — features and the label are both keyed by user, and a *training set* (next step) joins them on that key into `(features, label)` rows. **How they differ:** features are model *inputs* served online for live scoring; the label is the *output* used only for offline training; the entity is neither — it's the identity that ties them together.

In [7]:
@ff.entity
class User:
    avg_transactions = ff.Feature(
        average_user_transaction[["user_id", "avg_transaction_amt"]],
        variant="quickstart",
        type=ff.Float32,
        inference_store=redis,
    )
    fraudulent = ff.Label(
        transactions[["customerid", "isfraud"]],
        variant="quickstart",
        type=ff.Bool,
    )

## Register a training set

A training set joins features to a label on the entity key, giving one reproducible source of truth for model training — built from the *same* feature definitions that serve online, so there's no training-serving skew.

In [8]:
ff.register_training_set(
    name="fraud_training",
    variant="quickstart",
    label=("fraudulent", "quickstart"),
    features=[("avg_transactions", "quickstart")],
)

TrainingSetVariant(name='fraud_training', owner='default_owner', label=('fraudulent', 'quickstart'), features=[('avg_transactions', 'quickstart')], description='', variant='quickstart', feature_lags=[], tags=[], properties={}, created=None, schedule='', schedule_obj=None, provider='', status='NO_STATUS', error=None, server_status=None, resource_snowflake_config=None, type=<TrainingSetType.DYNAMIC: 1>)

## Apply the definitions

`client.apply()` registers everything with the coordinator and kicks off materialization: the SQL transformation runs in Postgres and the resulting feature values are pushed into Redis. `asynchronous=False` blocks until materialization finishes.

In [9]:
# NBVAL_SKIP
client = ff.Client(host=FEATUREFORM_HOST, insecure=True)
client.apply(asynchronous=False, verbose=True)

Applying Run: 2026-07-22t13-39-20
Creating User default_owner 
Creating Provider redis-quickstart 
Creating Provider postgres-quickstart 
Creating Source Variant transactions quickstart
Creating Source Variant average_user_transaction quickstart
Creating Entity user 
Creating Feature Variant avg_transactions quickstart
Creating Label Variant fraudulent quickstart
Creating Trainingset Variant fraud_training quickstart



UserWarning: install "ipywidgets" for Jupyter support

## Serve features from the Redis online store

Now for the payoff: request a feature for a single entity key. This read is served straight from Redis, so it returns in milliseconds — the pattern you'd put behind a real-time fraud model. The entity dict is keyed by the lowercased entity class name (`user`).

*Why this matters:* this is the exact same feature, by name and variant, that the training set below is built from — but fetched from Redis in the time budget of a live request. Define once, serve everywhere: the model scores production traffic on precisely the values it was trained on.

In [10]:
# NBVAL_SKIP
avg_txn = client.features(
    [("avg_transactions", "quickstart")],
    {"user": "C1214240"},
)
print("avg_transactions for user C1214240:", avg_txn)

avg_transactions for user C1214240: [319.0]


### Benchmark the online read

Serving from Redis is the whole point of an online store. A single feature lookup should land in the low-millisecond range end-to-end (gRPC round-trip + Redis read).

In [11]:
# NBVAL_SKIP
%timeit client.features([("avg_transactions", "quickstart")], {"user": "C1214240"})

1.27 ms ± 179 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


## Build a training set from the same definitions

The offline side reuses the exact feature/label definitions. The dataset is iterable and streams rows of `(features, label)`.

In [12]:
# NBVAL_SKIP
dataset = client.training_set("fraud_training", "quickstart")

for i, row in enumerate(dataset):
    print(row.features(), "->", row.label())
    if i >= 4:
        break

[[650.]] -> [False]
[[234.]] -> [ True]
[[1.]] -> [ True]
[[370.]] -> [False]
[[47.]] -> [ True]


## Inspect the feature values in Redis

Featureform materializes features into the online store, so we can connect to the quickstart Redis (published on `localhost:6379`) and confirm the keys landed. Values are stored under Featureform-encoded keys, so they won't be human-readable, but you'll see them populated.

In [13]:
# NBVAL_SKIP
from redis import Redis

# The quickstart Redis is published on the host at localhost:6379.
redis_client = Redis(host="localhost", port=6379, password="")
keys = redis_client.keys()
print(f"{len(keys)} keys in Redis; first 10:")
for k in keys[:10]:
    print(k)

2 keys in Redis; first 10:
b'{"Prefix":"Featureform_table__","Feature":"avg_transactions","Variant":"quickstart"}'
b'Featureform_table____tables'


## Cleanup

Stop and remove the quickstart containers when you're done.

In [14]:
# NBVAL_SKIP
!featureform stop docker

I0000 00:00:1784752778.502483 1749902 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


Tearing down Featureform on Docker
Stopping containers...
	Stopping featureform container
	Stopping quickstart-postgres container
	Stopping quickstart-redis container
Container quickstart-clickhouse not found. Skipping...


## Learn more

- [Redis Feature Form docs](https://redis.io/docs/latest/develop/ai/featureform/)
- [Featureform documentation](https://docs.featureform.com/)
- [Feast + Redis credit scoring recipe](./00_feast_credit_score.ipynb) — the other feature-store pattern in this repo